In [1]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

In [2]:
df = pd.read_csv("raw_gait_data.csv")

print("Original rows:", len(df))
df.head()

Original rows: 12000


,timestamp,L1,L2,L3,L4,L5,L6,L7,L8,R1,...,acc_z_left,gyro_x_left,gyro_y_left,gyro_z_left,acc_x_right,acc_y_right,acc_z_right,gyro_x_right,gyro_y_right,gyro_z_right
0,0.00,2871,2122,3095,2850,1870,0,2891,3579,367,...,9.673,0.046,0.030,-0.008,0.934,-2.911,9.680,0.006,0.021,0.012
1,0.01,2881,2126,3086,2855,1878,0,2885,3580,339,...,9.641,0.046,0.035,-0.006,0.943,-2.904,9.668,0.009,0.023,0.015
2,0.02,2875,2119,3071,2859,1886,0,2883,3583,327,...,9.639,0.040,0.027,-0.009,0.917,-2.940,9.673,0.011,0.026,0.014
3,0.03,2873,2128,3071,2854,1878,0,2883,3583,315,...,9.740,0.044,0.029,-0.003,0.910,-2.954,9.675,0.012,0.020,0.013
4,0.04,2873,2124,3060,2849,1875,0,2889,3583,309,...,9.680,0.047,0.035,-0.010,0.883,-2.947,9.656,0.016,0.020,0.007


In [3]:
TRIM_START = 100
TRIM_END = 100

df = df.iloc[TRIM_START : len(df) - TRIM_END]

print("After trimming:", len(df))

After trimming: 11800


In [4]:
SAMPLING_RATE = 100

WINDOW_SIZE = 400      # 4 sec window
STEP_SIZE = 200        # overlap

PRESSURE_THRESHOLD = 400

In [5]:
def extract_features(window):

    features = {}

    L = window[['L1','L2','L3','L4','L5','L6','L7','L8']].values
    R = window[['R1','R2','R3','R4','R5','R6','R7','R8']].values

    L_flat = L.flatten()
    R_flat = R.flatten()

    # Mean Pressure
    features['mean_pressure_left'] = np.mean(L_flat)
    features['mean_pressure_right'] = np.mean(R_flat)

    # Peak Pressure
    features['peak_pressure_left'] = np.max(L_flat)
    features['peak_pressure_right'] = np.max(R_flat)

    # Variance
    features['pressure_variance_left'] = np.var(L_flat)
    features['pressure_variance_right'] = np.var(R_flat)

    # Heel Toe Ratio
    heel_left = window['L8'].mean()
    heel_right = window['R8'].mean()

    toe_left = window['L1'].mean()
    toe_right = window['R1'].mean()

    features['heel_to_toe_ratio_left'] = np.clip(heel_left/(toe_left+1e-6),0,1.8)
    features['heel_to_toe_ratio_right'] = np.clip(heel_right/(toe_right+1e-6),0,1.8)

    # Contact Area
    features['contact_area_left'] = np.sum(L > PRESSURE_THRESHOLD)
    features['contact_area_right'] = np.sum(R > PRESSURE_THRESHOLD)

    # Impact Force
    features['impact_force_left'] = features['peak_pressure_left'] - features['mean_pressure_left']
    features['impact_force_right'] = features['peak_pressure_right'] - features['mean_pressure_right']

    # IMU Features
    acc_left = window[['acc_x_left','acc_y_left','acc_z_left']].values
    acc_right = window[['acc_x_right','acc_y_right','acc_z_right']].values

    gyro_left = window[['gyro_x_left','gyro_y_left','gyro_z_left']].values
    gyro_right = window[['gyro_x_right','gyro_y_right','gyro_z_right']].values

    features['imu_acc_mean_left'] = np.mean(acc_left)
    features['imu_acc_mean_right'] = np.mean(acc_right)

    features['imu_acc_std_left'] = np.std(acc_left)
    features['imu_acc_std_right'] = np.std(acc_right)

    features['imu_gyro_mean_left'] = np.mean(gyro_left)
    features['imu_gyro_mean_right'] = np.mean(gyro_right)

    features['imu_gyro_std_left'] = np.std(gyro_left)
    features['imu_gyro_std_right'] = np.std(gyro_right)

    # Step Detection (heel sensors)
    heel_signal = window['L8'].values + window['R8'].values

    peaks,_ = find_peaks(
        heel_signal,
        height=PRESSURE_THRESHOLD,
        distance=SAMPLING_RATE//2
    )

    if len(peaks) > 1:

        stride_samples = np.diff(peaks)
        stride_times = stride_samples / SAMPLING_RATE

        features['stride_time_mean'] = np.mean(stride_times)
        features['stride_time_std'] = np.std(stride_times)

        window_time = WINDOW_SIZE / SAMPLING_RATE
        steps = len(peaks)

        features['cadence'] = (steps / window_time) * 60

    else:

        features['stride_time_mean'] = 0
        features['stride_time_std'] = 0
        features['cadence'] = 0

    # Step Symmetry
    left_contact = np.sum(window['L8'] > PRESSURE_THRESHOLD)
    right_contact = np.sum(window['R8'] > PRESSURE_THRESHOLD)

    features['step_symmetry'] = left_contact/(right_contact+1e-6)

    # Force Symmetry
    features['force_symmetry'] = features['mean_pressure_left']/(features['mean_pressure_right']+1e-6)

    return features

In [6]:
rows = []

for i in range(0, len(df) - WINDOW_SIZE, STEP_SIZE):

    window = df.iloc[i : i + WINDOW_SIZE]

    features = extract_features(window)

    rows.append(features)

In [7]:
final_df = pd.DataFrame(rows)

In [8]:
final_df = final_df[[
    'cadence',
    'stride_time_mean',
    'stride_time_std',
    'mean_pressure_left',
    'mean_pressure_right',
    'peak_pressure_left',
    'peak_pressure_right',
    'pressure_variance_left',
    'pressure_variance_right',
    'heel_to_toe_ratio_left',
    'heel_to_toe_ratio_right',
    'contact_area_left',
    'contact_area_right',
    'impact_force_left',
    'impact_force_right',
    'imu_acc_mean_left',
    'imu_acc_mean_right',
    'imu_acc_std_left',
    'imu_acc_std_right',
    'imu_gyro_mean_left',
    'imu_gyro_mean_right',
    'imu_gyro_std_left',
    'imu_gyro_std_right',
    'step_symmetry',
    'force_symmetry'
]]

In [9]:
final_df.to_csv("prediction_input_dataset.csv", index=False)

print("Dataset ready for prediction ✅")

Dataset ready for prediction ✅
